[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sanemat/uol-fp/blob/main/proto1/pipeline.ipynb)

# Methodology Extraction Pipeline

Extracts structured methodology from a research paper.

Output format:
```json
{
  "Design": "experiment",
  "Method": ["BERT"],
  "Data": ["MNIST"],
  "Evaluation": ["accuracy"]
}
```

**Run in order: Cell 1 → 2 → 3 → 4 → 5 → 6 → 7**

## Setup

In [ ]:
!pip install transformers torch pdfplumber pymupdf requests

In [ ]:
import json
import re
from dataclasses import dataclass, field
from enum import Enum

import torch
from transformers import AutoModel, AutoTokenizer

print("Setup complete.")

## Data Models

In [ ]:
class DesignType(str, Enum):
    EXPERIMENT = "experiment"
    SURVEY = "survey"
    CASE_STUDY = "case_study"
    THEORETICAL = "theoretical"
    ALGORITHM_DEVELOPMENT = "algorithm_development"
    UNKNOWN = "unknown"


@dataclass
class CandidateWithContext:
    candidate: str
    sentence: str
    section: str = "unknown"
    source_paper: str = ""


@dataclass
class MethodologyProfile:
    design: DesignType = DesignType.UNKNOWN
    method: list = field(default_factory=list)
    task: list = field(default_factory=list)
    data: list = field(default_factory=list)
    evaluation: list = field(default_factory=list)

    def to_dict(self):
        result = {
            "Design": self.design.value,
            "Method": self.method,
            "Task": self.task,
        }
        optional = {}
        if self.data:
            optional["Data"] = self.data
        if self.evaluation:
            optional["Evaluation"] = self.evaluation
        if optional:
            result["Optional"] = optional
        return result


print("Models ready.")

## Step 0 — Load PDF

Upload a PDF file and extract text.
Skip this cell if you want to paste text manually in the next cell.

In [ ]:
import io

import pdfplumber
from google.colab import files

uploaded = files.upload()  # opens a file picker dialog
filename = next(iter(uploaded))

with pdfplumber.open(io.BytesIO(uploaded[filename])) as pdf:
    paper_text = "\n".join(page.extract_text() or "" for page in pdf.pages)

print(f"Loaded: {filename} ({len(paper_text)} chars)")
print(paper_text[:300])

## PDF Extraction Comparison (T8)

Compare PyMuPDF vs GROBID on the uploaded PDF.
Run both cells, then check which output is cleaner.


### Approach A — PyMuPDF


In [ ]:
import fitz  # pymupdf


def extract_pymupdf(pdf_bytes: bytes) -> str:
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    parts = []
    for page in doc:
        blocks = page.get_text("blocks", sort=True)
        for block in blocks:
            _x0, _y0, _x1, _y1, text, _block_no, block_type = block
            if block_type == 0 and text.strip():
                parts.append(text.strip())
    return "\n".join(parts)


text_pymupdf = extract_pymupdf(uploaded[filename])

words_a = text_pymupdf.split()
long_a = [w for w in words_a if len(w) > 25 and not w.startswith("http")]
print(f"PyMuPDF: {len(text_pymupdf)} chars, {len(long_a)} long words")
print()
print(text_pymupdf[:800])

### Approach B — GROBID


In [ ]:
from xml.etree import ElementTree as ET

import requests

GROBID_URL = "https://kermitt2-grobid.hf.space/api/processFulltextDocument"
NS = {"tei": "http://www.tei-c.org/ns/1.0"}


def extract_grobid(pdf_bytes: bytes) -> dict:
    resp = requests.post(
        GROBID_URL,
        files={"input": ("paper.pdf", pdf_bytes, "application/pdf")},
        data={"consolidateHeader": "0"},
        timeout=120,
    )
    resp.raise_for_status()
    root = ET.fromstring(resp.text)

    abstract = " ".join(
        (p.text or "") for p in root.findall(".//tei:abstract//tei:p", NS)
    ).strip()

    sections = []
    for div in root.findall(".//tei:body//tei:div", NS):
        head = div.findtext("tei:head", namespaces=NS) or "unknown"
        body = " ".join((p.text or "") for p in div.findall("tei:p", NS)).strip()
        if body:
            sections.append({"section": head, "text": body})

    return {"abstract": abstract, "sections": sections}


grobid_result = extract_grobid(uploaded[filename])

text_grobid = (
    grobid_result["abstract"]
    + "\n"
    + "\n".join(f"[{s['section']}] {s['text']}" for s in grobid_result["sections"])
)

words_b = text_grobid.split()
long_b = [w for w in words_b if len(w) > 25 and not w.startswith("http")]
print(f"GROBID: {len(text_grobid)} chars, {len(long_b)} long words")
print(f"Sections found: {[s['section'] for s in grobid_result['sections']]}")
print()
print(text_grobid[:800])

### Comparison Checklist


In [ ]:
print("=== Comparison ===")
print(f"PyMuPDF : {len(text_pymupdf):6d} chars | long words: {len(long_a)}")
print(f"GROBID  : {len(text_grobid):6d} chars | long words: {len(long_b)}")
print()
print("Checklist (check manually):")
print("  [ ] Spaces correct?")
print("  [ ] Sentence order natural?")
print("  [ ] References removed?")
print("  [ ] Section names available?")
print("  [ ] Tables separated from body?")
print()
print("Set paper_text = text_pymupdf  OR  text_grobid in the next cell.")

### Choose extraction result

Set `paper_text` to the better output before running Step 1.


In [ ]:
# Choose one:
paper_text = text_pymupdf
# paper_text = text_grobid

print(f"Using {len(paper_text)} chars as paper_text.")

## Input — Paste text manually (skip if Step 0 ran)

In [ ]:
# Paste your paper text (abstract or full text)
paper_text = """
We propose a novel BERT-based model for text classification.
We fine-tune BERT on the SST-2 dataset and evaluate using accuracy and F1 score.
Our experiments show that the proposed method outperforms the CNN baseline.
"""

print(f"Input text ({len(paper_text)} chars):")
print(paper_text[:200])

## Text Quality Check


In [ ]:
print("=== Raw text sample (first 1000 chars) ===")
print(paper_text[:1000])
print()

words = paper_text.split()
long_words = [w for w in words if len(w) > 25 and not w.startswith("http")]
print(f"Words > 25 chars (likely broken PDF spacing): {len(long_words)}")
for w in long_words[:10]:
    print(f"  {w}")

## Section Filter


In [ ]:
def filter_paper_text(text: str) -> str:
    ref_pattern = re.compile(
        r"^\s*(References|Bibliography|REFERENCES|BIBLIOGRAPHY)\s*$",
        re.MULTILINE,
    )
    m = ref_pattern.search(text)
    if m:
        text = text[: m.start()]

    caption_pattern = re.compile(
        r"^(Figure|Fig\.|Table|TABLE)\s+\d+.*$",
        re.MULTILINE | re.IGNORECASE,
    )
    text = caption_pattern.sub("", text)
    return text


filtered_text = filter_paper_text(paper_text)
print(f"Original: {len(paper_text)} chars → Filtered: {len(filtered_text)} chars")

## Step 1 — Candidate Extraction (SciBERT NER)

In [ ]:
# Load SciBERT
MODEL_NAME = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()


def get_sentence_embeddings(text: str) -> list[tuple[str, list[float]]]:
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    results = []
    for sent in sentences:
        inputs = tokenizer(sent, return_tensors="pt", truncation=True, max_length=128)
        with torch.no_grad():
            outputs = model(**inputs)
        embedding = outputs.last_hidden_state[0, 0, :].tolist()
        results.append((sent, embedding))
    return results


_STOP = {
    "We",
    "Our",
    "The",
    "This",
    "In",
    "A",
    "An",
    "To",
    "For",
    "On",
    "Is",
    "It",
    "At",
    "By",
    "As",
    "Of",
    "Be",
    "Are",
    "Was",
    "Has",
    "Have",
    "From",
    "With",
    "That",
    "Which",
    "These",
    "Those",
    "Also",
    "Such",
    "Both",
    "Each",
}


def extract_candidates(text: str) -> list[CandidateWithContext]:
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    patterns = [
        r"\b[A-Z][A-Za-z0-9]*(?:-[A-Za-z0-9]+)*\b",
        r"\b[a-z]+-\d+\b",
        r"\b(?:accuracy|f1|precision|recall|bleu|rouge|auc)\b",
        r"\b(?:dataset|corpus|benchmark)\b",
    ]
    results: list[CandidateWithContext] = []
    seen: set[str] = set()
    for sent in sentences:
        for pattern in patterns:
            for match in re.finditer(pattern, sent, re.IGNORECASE):
                term = match.group().strip()
                if (
                    len(term) < 3
                    or len(term) > 40
                    or term in _STOP
                    or term.lower() in {s.lower() for s in _STOP}
                    or term in seen
                ):
                    continue
                seen.add(term)
                results.append(CandidateWithContext(candidate=term, sentence=sent))
    return results


sentence_embeddings = get_sentence_embeddings(filtered_text)
candidates_with_ctx = extract_candidates(filtered_text)

print(f"Sentences: {len(sentence_embeddings)}")
print(f"Candidates: {len(candidates_with_ctx)}")

## Step 2 — Role Classification

In [ ]:
class Role(str, Enum):
    METHOD = "Method"
    TASK = "Task"
    DATA = "Data"
    EVALUATION = "Evaluation"
    OTHER = "Other"


_KNOWN_METHODS = {
    "bert",
    "gpt",
    "roberta",
    "xlnet",
    "t5",
    "gpt-2",
    "gpt-3",
    "lstm",
    "cnn",
    "rnn",
    "transformer",
    "attention",
    "svm",
    "random forest",
    "k-means",
    "resnet",
    "vgg",
    "scibert",
}
_KNOWN_TASKS = {
    "question answering",
    "text classification",
    "image classification",
    "named entity recognition",
    "machine translation",
    "summarization",
    "sentiment analysis",
    "relation extraction",
    "coreference resolution",
}
_KNOWN_DATA = {
    "mnist",
    "cifar",
    "squad",
    "glue",
    "superglue",
    "imagenet",
    "sst-2",
    "sst-1",
    "conll",
    "wikitext",
    "bookcorpus",
    "imdb",
    "yelp",
    "amazon",
    "snli",
    "mnli",
}
_KNOWN_EVAL = {
    "accuracy",
    "f1",
    "precision",
    "recall",
    "bleu",
    "rouge",
    "auc",
    "map",
    "ndcg",
    "perplexity",
    "em",
    "exact match",
}

_METHOD_RE = re.compile(
    r"\b(neural|network|model|algorithm|architecture|classifier|proposed)\b",
    re.IGNORECASE,
)
_TASK_RE = re.compile(
    r"\b(task|problem|classification|recognition|detection|generation|translation)\b",
    re.IGNORECASE,
)
_DATA_RE = re.compile(
    r"\b(dataset|corpus|benchmark|collection|training|test)\b",
    re.IGNORECASE,
)
_EVAL_RE = re.compile(
    r"\b(score|metric|performance|rate|result)\b",
    re.IGNORECASE,
)


def classify_role(candidate: str, context: str = "") -> Role:
    low = candidate.lower()
    if low in _KNOWN_EVAL or _EVAL_RE.search(low):
        return Role.EVALUATION
    if low in _KNOWN_DATA or _DATA_RE.search(context.lower()):
        return Role.DATA
    if low in _KNOWN_TASKS or _TASK_RE.search(context.lower()):
        return Role.TASK
    if low in _KNOWN_METHODS or _METHOD_RE.search(context.lower()):
        return Role.METHOD
    return Role.OTHER


classified: dict[str, Role] = {}
for cwc in candidates_with_ctx:
    classified[cwc.candidate] = classify_role(cwc.candidate, cwc.sentence)

for term, role in sorted(classified.items()):
    print(f"  {role.value:12s}  {term}")

## Candidate Log (Error Analysis)


In [ ]:
log = []
for cwc in candidates_with_ctx:
    log.append(
        {
            "candidate": cwc.candidate,
            "role": classified.get(cwc.candidate, Role.OTHER).value,
            "sentence": cwc.sentence,
            "section": cwc.section,
        }
    )

print(json.dumps(log, indent=2))

## Step 3 — Design Detection

In [ ]:
_DESIGN_PATTERNS: list[tuple[DesignType, list[str]]] = [
    (
        DesignType.EXPERIMENT,
        [r"\bexperiment\w*\b", r"\buser study\b", r"\bablation\b"],
    ),
    (
        DesignType.SURVEY,
        [r"\bsurvey\b", r"\bliterature review\b", r"\bsystematic review\b"],
    ),
    (DesignType.CASE_STUDY, [r"\bcase study\b", r"\bcase studies\b"]),
    (DesignType.THEORETICAL, [r"\btheor\w+\b", r"\bproof\b", r"\bformal\w*\b"]),
    (
        DesignType.ALGORITHM_DEVELOPMENT,
        [r"\balgorithm\w*\b", r"\barchitecture\b", r"\bpropose\w*\b", r"\bnovel\b"],
    ),
]


def detect_design(text: str) -> DesignType:
    text_lower = text.lower()
    scores: dict[DesignType, int] = {}
    for design_type, patterns in _DESIGN_PATTERNS:
        count = sum(len(re.findall(p, text_lower)) for p in patterns)
        if count > 0:
            scores[design_type] = count
    if not scores:
        return DesignType.UNKNOWN
    return max(scores, key=lambda k: scores[k])


design = detect_design(paper_text)
print(f"Design: {design.value}")

## Step 4 — Build JSON Output

In [ ]:
profile = MethodologyProfile(
    design=design,
    method=[t for t, r in classified.items() if r == Role.METHOD],
    task=[t for t, r in classified.items() if r == Role.TASK],
    data=[t for t, r in classified.items() if r == Role.DATA],
    evaluation=[t for t, r in classified.items() if r == Role.EVALUATION],
)

print(json.dumps(profile.to_dict(), indent=2))

## Step 5 — Consistency Checking

In [ ]:
@dataclass
class ConsistencyResult:
    is_valid: bool
    warnings: list


def check_consistency(profile: MethodologyProfile) -> ConsistencyResult:
    warnings = []
    if profile.design == DesignType.EXPERIMENT:
        if not profile.task:
            warnings.append("Experimental paper without Task is weak.")
        if not profile.method:
            warnings.append("Experimental paper without Method is weak.")
    if profile.method and not profile.task:
        warnings.append("Method without Task may be incomplete.")
    if profile.design == DesignType.THEORETICAL and profile.evaluation:
        warnings.append("Theoretical design with Evaluation may be a mismatch.")
    return ConsistencyResult(is_valid=len(warnings) == 0, warnings=warnings)


result = check_consistency(profile)
print(f"Valid: {result.is_valid}")
for w in result.warnings:
    print(f"  WARNING: {w}")
if result.is_valid:
    print("  No issues found.")